In [1]:
from marubatsu import Marubatsu
import random

def playout(self, move=None):
    board = self.board.save()
    turn = self.turn
    move_count = self.move_count
    status = self.status
    while status == self.PLAYING:
        if move is None:
            move = random.choice(self.calc_legal_moves())
        self.board.setmark_by_move(move, turn)
        last_turn = turn
        turn = self.CROSS if turn == self.CIRCLE else self.CIRCLE
        move_count += 1
        status = self.board.judge(last_turn, move, move_count)
        move = None
    self.board.load(board)
    return status

Marubatsu.playout = playout

In [ ]:
from ai import dprint
from random import choice
from time import perf_counter

def ai_pmc(mb, pnum=10000, timelimit=None, debug=False, analyze=False, *args, **kwargs):
    if mb.move_count == 8:
        best_move = mb.calc_legal_moves()[0]
        if analyze:
            return {
                "candidate": [mb.board.move_to_xy(best_move)],
                "ratio_by_move": {},
                "playout num": 0,
                "score_by_move": {best_move: 1},                
            }
        else:
            return best_move
    
    if timelimit is not None:   
        starttime = perf_counter()
        timelimit_pc = starttime + timelimit
    result = {}
    legal_moves = mb.calc_legal_moves()
    for move in legal_moves:
        result[move] = {
            mb.CIRCLE: 0,
            mb.CROSS: 0,
            mb.DRAW: 0,
        }
    playout_num = 0
    for _ in range(pnum):
        if timelimit is not None and perf_counter() > timelimit_pc:
            break
        for move in legal_moves:
            result[move][mb.playout(move)] += 1
        playout_num += 1

    best_moves = []
    best_movesxy = []
    best_ratio = (-1, 0)
    if analyze:
        ratio_by_move = {}
    for move, count in result.items():
        totalcount = max(1, playout_num)
        winratio = count[mb.turn] / totalcount
        drawratio = count[mb.DRAW] / totalcount
        movexy = mb.board.move_to_xy(move)
        dprint(debug, "=" * 50)
        dprint(debug, f"move {movexy}")
        dprint(debug, f"ratio      win: {winratio:.3f} draw {drawratio:.3f}")
        dprint(debug, f"best ratio win: {best_ratio[0]:.3f} draw {best_ratio[1]:.3f}", )
        if best_ratio is None or winratio > best_ratio[0] or (winratio == best_ratio[0] and drawratio > best_ratio[1]):
            best_ratio = (winratio, drawratio)
            best_moves = [move]
            best_movesxy = [movexy]
            dprint(debug, "UPDATE")
            dprint(debug, f"  best score {best_ratio}")
            dprint(debug, f"  best moves {best_movesxy}")
        elif winratio == best_ratio[0] and drawratio == best_ratio[1]:
            best_moves.append(move)
            best_movesxy.append(movexy)
            dprint(debug, "APPEND")
            dprint(debug, f"  best moves {best_movesxy}")
        if analyze:
            ratio_by_move[movexy] = (winratio, drawratio)
    if analyze:
        score_by_move = {mb.board.xy_to_move(x, y): round(winratio, 3) 
                         for (x, y), (winratio, drawratio) in ratio_by_move.items() }
        if mb.status == mb.PLAYING and max(score_by_move.values()) == 0:
            score_by_move = {mb.board.xy_to_move(x, y): round(drawratio, 3) 
                             for (x, y), (winratio, drawratio) in ratio_by_move.items() }         
        score_by_movexy = {
            mb.board.move_to_xy(move): score 
            for move, score in score_by_move.items()
        } 
        return {
            "candidate": best_movesxy,
            "ratio_by_move": ratio_by_move,
            "playout num": playout_num,
            "score_by_move": score_by_move,
            "score_by_movexy": score_by_movexy,
        }
    else:
        return choice(best_moves) 

In [3]:
mb = Marubatsu()

ai_pmc(mb, pnum=10000, analyze=True, debug=True)

move (0, 0)
ratio      win: 0.601 draw 0.128
best ratio win: -1.000 draw 0.000
UPDATE
  best score (0.6006, 0.1275)
  best moves [(0, 0)]
move (0, 1)
ratio      win: 0.531 draw 0.132
best ratio win: 0.601 draw 0.128
move (0, 2)
ratio      win: 0.603 draw 0.128
best ratio win: 0.601 draw 0.128
UPDATE
  best score (0.6034, 0.1281)
  best moves [(0, 2)]
move (1, 0)
ratio      win: 0.542 draw 0.126
best ratio win: 0.603 draw 0.128
move (1, 1)
ratio      win: 0.697 draw 0.112
best ratio win: 0.603 draw 0.128
UPDATE
  best score (0.6972, 0.1121)
  best moves [(1, 1)]
move (1, 2)
ratio      win: 0.536 draw 0.136
best ratio win: 0.697 draw 0.112
move (2, 0)
ratio      win: 0.604 draw 0.130
best ratio win: 0.697 draw 0.112
move (2, 1)
ratio      win: 0.534 draw 0.130
best ratio win: 0.697 draw 0.112
move (2, 2)
ratio      win: 0.606 draw 0.128
best ratio win: 0.697 draw 0.112


{'candidate': [(1, 1)],
 'ratio_by_move': {(0, 0): (0.6006, 0.1275),
  (0, 1): (0.5311, 0.1318),
  (0, 2): (0.6034, 0.1281),
  (1, 0): (0.5417, 0.1259),
  (1, 1): (0.6972, 0.1121),
  (1, 2): (0.536, 0.1357),
  (2, 0): (0.6038, 0.1295),
  (2, 1): (0.5337, 0.1296),
  (2, 2): (0.6057, 0.1277)},
 'playout num': 10000,
 'score_by_move': {1: 0.601,
  2: 0.531,
  4: 0.603,
  8: 0.542,
  16: 0.697,
  32: 0.536,
  64: 0.604,
  128: 0.534,
  256: 0.606},
 'score_by_movexy': {(0, 0): 0.601,
  (0, 1): 0.531,
  (0, 2): 0.603,
  (1, 0): 0.542,
  (1, 1): 0.697,
  (1, 2): 0.536,
  (2, 0): 0.604,
  (2, 1): 0.534,
  (2, 2): 0.606}}

In [4]:
from ai import ai_match, ai2s, ai14s

ai_match(ai=[ai_pmc, ai2s], params=[{"pnum": 100000000, "timelimit": 0.1}, {}], match_num=1000)
ai_match(ai=[ai_pmc, ai14s], params=[{"pnum": 100000000, "timelimit": 0.1}, {}], match_num=1000)

ai_pmc VS ai2s


100%|██████████| 1000/1000 [10:43<00:00,  1.55it/s]


count     win    lose    draw
o         989       0      11
x         895      88      17
total    1884      88      28

ratio     win    lose    draw
o       98.9%    0.0%    1.1%
x       89.5%    8.8%    1.7%
total   94.2%    4.4%    1.4%

ai_pmc VS ai14s


100%|██████████| 1000/1000 [11:42<00:00,  1.42it/s]

count     win    lose    draw
o           0       0    1000
x           0    1000       0
total       0    1000    1000

ratio     win    lose    draw
o        0.0%    0.0%  100.0%
x        0.0%  100.0%    0.0%
total    0.0%   50.0%   50.0%



[('count',
  [{'win': 0, 'lose': 0, 'draw': 1000},
   {'win': 0, 'lose': 1000, 'draw': 0},
   {'win': 0, 'lose': 1000, 'draw': 1000}],
  '7d'),
 ('ratio',
  [{'win': 0.0, 'lose': 0.0, 'draw': 1.0},
   {'win': 0.0, 'lose': 1.0, 'draw': 0.0},
   {'win': 0.0, 'lose': 0.5, 'draw': 0.5}],
  '7.1%')]

In [5]:
from collections import defaultdict

def ai_pmc2(mb, pnum=10000, timelimit=None, debug=False, analyze=False, *args, **kwargs):
    if mb.move_count == 8:
        best_move = mb.calc_legal_moves()[0]
        if analyze:
            return {
                "candidate": [mb.board.move_to_xy(best_move)],
                "ratio_by_move": {},
                "playout num": 0,
                "score_by_move": {best_move: 1},                
            }
        else:
            return best_move
    
    if timelimit is not None:   
        starttime = perf_counter()
        timelimit_pc = starttime + timelimit
    legal_moves = mb.calc_legal_moves()
    score_by_move = defaultdict(float)

    playout_num = 0
    for _ in range(pnum):
        if timelimit is not None and perf_counter() > timelimit_pc:
            break
        for move in legal_moves:
            status = mb.playout(move)
            if status == mb.turn:
                score_by_move[move] += 1
            elif status == mb.DRAW:
                score_by_move[move] += 0.5
        playout_num += 1
    for move in legal_moves:
        score_by_move[move] /= max(1, playout_num)

    best_moves = []
    best_movesxy = []
    best_score = 0
    for move, score in score_by_move.items():
        movexy = mb.board.move_to_xy(move)
        dprint(debug, "=" * 50)
        dprint(debug, f"move {movexy}")
        dprint(debug, f"score:      {score:.3f}")
        dprint(debug, f"best score: {best_score:.3f}")
        if score > best_score:
            best_score = score
            best_moves = [move]
            best_movesxy = [movexy]
            dprint(debug, "UPDATE")
            dprint(debug, f"  best score {best_score}")
            dprint(debug, f"  best moves {best_movesxy}")
        elif score == best_score:
            best_moves.append(move)
            best_movesxy.append(movexy)
            dprint(debug, "APPEND")
            dprint(debug, f"  best moves {best_movesxy}")
    if analyze:
        score_by_movexy = {
            mb.board.move_to_xy(move): score 
            for move, score in score_by_move.items()
        }
        return {
            "candidate": best_movesxy,
            "playout num": playout_num,
            "score_by_move": score_by_movexy
        }
    else:
        return choice(best_moves) 

In [6]:
ai_pmc2(mb, pnum=10000, analyze=True, debug=True)

move (0, 0)
score:      0.667
best score: 0.000
UPDATE
  best score 0.6668
  best moves [(0, 0)]
move (0, 1)
score:      0.603
best score: 0.667
move (0, 2)
score:      0.683
best score: 0.667
UPDATE
  best score 0.68345
  best moves [(0, 2)]
move (1, 0)
score:      0.599
best score: 0.683
move (1, 1)
score:      0.747
best score: 0.683
UPDATE
  best score 0.74665
  best moves [(1, 1)]
move (2, 1)
score:      0.600
best score: 0.747
move (1, 2)
score:      0.600
best score: 0.747
move (2, 0)
score:      0.673
best score: 0.747
move (2, 2)
score:      0.670
best score: 0.747


{'candidate': [(1, 1)],
 'playout num': 10000,
 'score_by_move': {(0, 0): 0.6668,
  (0, 1): 0.603,
  (0, 2): 0.68345,
  (1, 0): 0.59895,
  (1, 1): 0.74665,
  (2, 1): 0.5996,
  (1, 2): 0.59975,
  (2, 0): 0.67295,
  (2, 2): 0.67}}

In [8]:
ai_match(ai=[ai_pmc2, ai2s], params=[{"pnum": 100000000, "timelimit": 0.1}, {}], match_num=1000)
ai_match(ai=[ai_pmc2, ai14s], params=[{"pnum": 100000000, "timelimit": 0.1}, {}], match_num=1000)

ai_pmc2 VS ai2s


100%|██████████| 1000/1000 [10:42<00:00,  1.56it/s]


count     win    lose    draw
o         987       0      13
x         888      42      70
total    1875      42      83

ratio     win    lose    draw
o       98.7%    0.0%    1.3%
x       88.8%    4.2%    7.0%
total   93.8%    2.1%    4.2%

ai_pmc2 VS ai14s


100%|██████████| 1000/1000 [12:32<00:00,  1.33it/s]

count     win    lose    draw
o           0       0    1000
x           0     497     503
total       0     497    1503

ratio     win    lose    draw
o        0.0%    0.0%  100.0%
x        0.0%   49.7%   50.3%
total    0.0%   24.9%   75.1%



[('count',
  [{'win': 0, 'lose': 0, 'draw': 1000},
   {'win': 0, 'lose': 497, 'draw': 503},
   {'win': 0, 'lose': 497, 'draw': 1503}],
  '7d'),
 ('ratio',
  [{'win': 0.0, 'lose': 0.0, 'draw': 1.0},
   {'win': 0.0, 'lose': 0.497, 'draw': 0.503},
   {'win': 0.0, 'lose': 0.2485, 'draw': 0.7515}],
  '7.1%')]